# Interactive Optical Parameter Explorer

Based on fit_batoid.py approach - fits AOS DOFs + atmospheric seeing moments.
Sliders update the batoid predictions in real-time.

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '10'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from matplotlib.collections import PatchCollection
from ipywidgets import FloatSlider, VBox, HBox, Output, interactive_output
import ipywidgets as widgets
from IPython.display import display

from fit_optics import (
    get_telescope, launch_rays, WCS, BatoidFitter,
    load_ccd_geometry, plot_ccd_polygons, AOS_DOF_INDICES
)

In [ ]:
# Load data and setup
band = 'r'
telescope = get_telescope(band)
data = pd.read_parquet('iq_dat.parquet')

# Apply radial cut
rcut = 300
r = np.sqrt(data.xfp**2 + data.yfp**2)
data = data[r < rcut].reset_index(drop=True)

# Compute WCS (focal plane to tangent plane angles)
wcs = WCS(telescope, band)
ax, ay = wcs(data.xfp.to_numpy(), data.yfp.to_numpy())
data['ax'] = ax
data['ay'] = ay

geometry = load_ccd_geometry()
print(f"Loaded {len(data)} CCDs")

In [ ]:
# Pre-compute observed quantities (don't change)
obs_mxx = data['mxx'].to_numpy()
obs_myy = data['myy'].to_numpy()
obs_mxy = data['mxy'].to_numpy()
obs_T = obs_mxx + obs_myy
obs_e1 = (obs_mxx - obs_myy) / obs_T
obs_e2 = 2 * obs_mxy / obs_T

# Mean-subtract for spatial patterns
obs_dT = obs_T - np.mean(obs_T)
obs_de1 = obs_e1 - np.mean(obs_e1)
obs_de2 = obs_e2 - np.mean(obs_e2)

detectors = data['det'].to_numpy() if 'det' in data.columns else np.arange(len(data))
xfp = data['xfp'].to_numpy()
yfp = data['yfp'].to_numpy()

vmin_dT, vmax_dT = -0.5, 0.5
vmin_de, vmax_de = -0.15, 0.15

In [ ]:
from batoid_rubin import LSSTBuilder

builder = LSSTBuilder(telescope)

def compute_batoid_moments(aos_dof, smxx, smyy, smxy):
    """Compute batoid moments for given AOS DOF."""
    tel = builder.with_aos_dof(aos_dof).build()
    spots = launch_rays(tel, band, data['ax'].to_numpy(), data['ay'].to_numpy())
    
    # Add seeing
    bat_mxx = spots.mxx.to_numpy() + smxx
    bat_myy = spots.myy.to_numpy() + smyy
    bat_mxy = spots.mxy.to_numpy() + smxy
    
    return bat_mxx, bat_myy, bat_mxy

def make_plot(cam_dz, cam_dx, cam_dy, cam_rx, cam_ry,
              m2_dz, m2_dx, m2_dy, m2_rx, m2_ry,
              smxx, smyy, smxy):
    
    # Build AOS DOF array
    aos_dof = np.zeros(50)
    aos_dof[AOS_DOF_INDICES['cam_dz']] = cam_dz
    aos_dof[AOS_DOF_INDICES['cam_dx']] = cam_dx
    aos_dof[AOS_DOF_INDICES['cam_dy']] = cam_dy
    aos_dof[AOS_DOF_INDICES['cam_rx']] = cam_rx
    aos_dof[AOS_DOF_INDICES['cam_ry']] = cam_ry
    aos_dof[AOS_DOF_INDICES['m2_dz']] = m2_dz
    aos_dof[AOS_DOF_INDICES['m2_dx']] = m2_dx
    aos_dof[AOS_DOF_INDICES['m2_dy']] = m2_dy
    aos_dof[AOS_DOF_INDICES['m2_rx']] = m2_rx
    aos_dof[AOS_DOF_INDICES['m2_ry']] = m2_ry
    
    # Compute batoid moments
    bat_mxx, bat_myy, bat_mxy = compute_batoid_moments(aos_dof, smxx, smyy, smxy)
    
    # Compute T, e1, e2
    bat_T = bat_mxx + bat_myy
    bat_e1 = (bat_mxx - bat_myy) / bat_T
    bat_e2 = 2 * bat_mxy / bat_T
    
    # Mean-subtract
    bat_dT = bat_T - np.nanmean(bat_T)
    bat_de1 = bat_e1 - np.nanmean(bat_e1)
    bat_de2 = bat_e2 - np.nanmean(bat_e2)
    
    # Residuals
    res_dT = obs_dT - bat_dT
    res_de1 = obs_de1 - bat_de1
    res_de2 = obs_de2 - bat_de2
    
    # Chi2
    chi2 = np.nansum(res_dT**2) + np.nansum(res_de1**2) + np.nansum(res_de2**2)
    
    # Correlations
    valid = np.isfinite(bat_dT) & np.isfinite(obs_dT)
    rho_T = np.corrcoef(obs_dT[valid], bat_dT[valid])[0, 1] if np.sum(valid) > 2 else 0
    rho_e1 = np.corrcoef(obs_de1[valid], bat_de1[valid])[0, 1] if np.sum(valid) > 2 else 0
    rho_e2 = np.corrcoef(obs_de2[valid], bat_de2[valid])[0, 1] if np.sum(valid) > 2 else 0
    
    # Plot
    fig, axes = plt.subplots(3, 3, figsize=(14, 12))
    lim = 350
    
    # Row 1: Observed
    for i, (val, title, vmin, vmax) in enumerate([
        (obs_dT, 'Observed dT', vmin_dT, vmax_dT),
        (obs_de1, 'Observed de1', vmin_de, vmax_de),
        (obs_de2, 'Observed de2', vmin_de, vmax_de),
    ]):
        ax = axes[0, i]
        if geometry:
            plot_ccd_polygons(ax, detectors, val, geometry, 'seismic', vmin, vmax)
        else:
            ax.scatter(xfp, yfp, c=val, s=40, cmap='seismic', vmin=vmin, vmax=vmax)
        ax.set_title(title)
        ax.set_aspect('equal')
        ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    
    # Row 2: Batoid
    for i, (val, title, vmin, vmax) in enumerate([
        (bat_dT, 'Batoid dT', vmin_dT, vmax_dT),
        (bat_de1, 'Batoid de1', vmin_de, vmax_de),
        (bat_de2, 'Batoid de2', vmin_de, vmax_de),
    ]):
        ax = axes[1, i]
        if geometry:
            plot_ccd_polygons(ax, detectors, val, geometry, 'seismic', vmin, vmax)
        else:
            ax.scatter(xfp, yfp, c=val, s=40, cmap='seismic', vmin=vmin, vmax=vmax)
        ax.set_title(title)
        ax.set_aspect('equal')
        ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    
    # Row 3: Residuals
    for i, (val, title, vmin, vmax) in enumerate([
        (res_dT, 'Residual dT', vmin_dT, vmax_dT),
        (res_de1, 'Residual de1', vmin_de, vmax_de),
        (res_de2, 'Residual de2', vmin_de, vmax_de),
    ]):
        ax = axes[2, i]
        if geometry:
            plot_ccd_polygons(ax, detectors, val, geometry, 'seismic', vmin, vmax)
        else:
            ax.scatter(xfp, yfp, c=val, s=40, cmap='seismic', vmin=vmin, vmax=vmax)
        ax.set_title(title)
        ax.set_aspect('equal')
        ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    
    fig.suptitle(f'Chi2={chi2:.1f} | Corr: dT={rho_T:.2f}, de1={rho_e1:.2f}, de2={rho_e2:.2f}',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# Create sliders
style = {'description_width': '100px'}
layout = widgets.Layout(width='280px')

# Camera DOF (microns for translations, arcsec for rotations)
cam_dz = FloatSlider(value=0, min=-100, max=100, step=2, description='cam_dz (um)', style=style, layout=layout, continuous_update=False)
cam_dx = FloatSlider(value=0, min=-100, max=100, step=2, description='cam_dx (um)', style=style, layout=layout, continuous_update=False)
cam_dy = FloatSlider(value=0, min=-100, max=100, step=2, description='cam_dy (um)', style=style, layout=layout, continuous_update=False)
cam_rx = FloatSlider(value=0, min=-5, max=5, step=0.1, description='cam_rx (")', style=style, layout=layout, continuous_update=False)
cam_ry = FloatSlider(value=0, min=-5, max=5, step=0.1, description='cam_ry (")', style=style, layout=layout, continuous_update=False)

# M2 DOF
m2_dz = FloatSlider(value=0, min=-100, max=100, step=2, description='m2_dz (um)', style=style, layout=layout, continuous_update=False)
m2_dx = FloatSlider(value=0, min=-100, max=100, step=2, description='m2_dx (um)', style=style, layout=layout, continuous_update=False)
m2_dy = FloatSlider(value=0, min=-100, max=100, step=2, description='m2_dy (um)', style=style, layout=layout, continuous_update=False)
m2_rx = FloatSlider(value=0, min=-5, max=5, step=0.1, description='m2_rx (")', style=style, layout=layout, continuous_update=False)
m2_ry = FloatSlider(value=0, min=-5, max=5, step=0.1, description='m2_ry (")', style=style, layout=layout, continuous_update=False)

# Seeing (pixels^2)
smxx = FloatSlider(value=1.3, min=0, max=3, step=0.1, description='smxx', style=style, layout=layout, continuous_update=False)
smyy = FloatSlider(value=1.3, min=0, max=3, step=0.1, description='smyy', style=style, layout=layout, continuous_update=False)
smxy = FloatSlider(value=0, min=-1, max=1, step=0.05, description='smxy', style=style, layout=layout, continuous_update=False)

# Layout
cam_box = VBox([widgets.HTML('<b>Camera</b>'), cam_dz, cam_dx, cam_dy, cam_rx, cam_ry])
m2_box = VBox([widgets.HTML('<b>M2</b>'), m2_dz, m2_dx, m2_dy, m2_rx, m2_ry])
seeing_box = VBox([widgets.HTML('<b>Seeing</b>'), smxx, smyy, smxy])

ui = HBox([cam_box, m2_box, seeing_box])

out = interactive_output(make_plot, {
    'cam_dz': cam_dz, 'cam_dx': cam_dx, 'cam_dy': cam_dy, 'cam_rx': cam_rx, 'cam_ry': cam_ry,
    'm2_dz': m2_dz, 'm2_dx': m2_dx, 'm2_dy': m2_dy, 'm2_rx': m2_rx, 'm2_ry': m2_ry,
    'smxx': smxx, 'smyy': smyy, 'smxy': smxy
})

display(ui, out)

In [ ]:
# Load fitted parameters and set sliders
import pickle

try:
    with open('fit_params.pkl', 'rb') as f:
        fit_params = pickle.load(f)
    print("Loaded fit_params.pkl:")
    for k, v in fit_params.items():
        print(f"  {k}: {v:.4f}")
    
    # Set sliders to fitted values
    if 'cam_dz' in fit_params: cam_dz.value = fit_params['cam_dz']
    if 'cam_dx' in fit_params: cam_dx.value = fit_params['cam_dx']
    if 'cam_dy' in fit_params: cam_dy.value = fit_params['cam_dy']
    if 'cam_rx' in fit_params: cam_rx.value = fit_params['cam_rx']
    if 'cam_ry' in fit_params: cam_ry.value = fit_params['cam_ry']
    if 'm2_dz' in fit_params: m2_dz.value = fit_params['m2_dz']
    if 'm2_dx' in fit_params: m2_dx.value = fit_params['m2_dx']
    if 'm2_dy' in fit_params: m2_dy.value = fit_params['m2_dy']
    if 'm2_rx' in fit_params: m2_rx.value = fit_params['m2_rx']
    if 'm2_ry' in fit_params: m2_ry.value = fit_params['m2_ry']
    if 'smxx' in fit_params: smxx.value = fit_params['smxx']
    if 'smyy' in fit_params: smyy.value = fit_params['smyy']
    if 'smxy' in fit_params: smxy.value = fit_params['smxy']
except FileNotFoundError:
    print("No fit_params.pkl found - using default values")